# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

All entities, including record sets, fields, and columns, are referenced by their `@id` fields as required by the FAIR² specification.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Access metadata directly as an object, not by subscripting or iterating.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display the dataset metadata
metadata_obj = dataset.metadata
print(f"Dataset title: {metadata_obj.name}")
print(f"Description: {metadata_obj.description}")
print(f"Identifier: {metadata_obj.identifier}")
print(f"Published: {metadata_obj.datePublished}")
print(f"License: {metadata_obj.license}")

# Optional: show keywords
print("Keywords:", getattr(metadata_obj, 'keywords', []))

## 2. Data Overview
Review available record sets and their fields, referencing them by their `@id`.

The Croissant schema provides the record sets with structured tables. We'll programmatically list the available record sets and show their fields.

In [ ]:
# List available record sets and their fields/columns
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    # Print fields or columns
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - {field.get('@id')} (name: {field.get('name', 'N/A')})")
            else:
                print(f"    - {field}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for column in columns:
            if isinstance(column, dict):
                print(f"    - {column.get('@id')} (name: {column.get('name', 'N/A')})")
            else:
                print(f"    - {column}")
    print()

# If there is at least one record set, load and print its first few records
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nFirst 3 records from RecordSet {first_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Reference each record set by its `@id`.

We'll build a dictionary of DataFrames for each record set.

In [ ]:
# Extract data from all record sets (by @id)
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    # Load records from the record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print columns and first rows from the first record set
if record_set_ids:
    sample_rs_id = record_set_ids[0]
    print(f"Columns for RecordSet {sample_rs_id}:")
    print(dataframes[sample_rs_id].columns.tolist())
    print("Sample data:")
    display(dataframes[sample_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, and group by key attributes. All fields are referenced by their `@id`.

Let's pick a numeric field and group field for the first record set if available.

In [ ]:
# Identify numeric field and group field
from pandas.api.types import is_numeric_dtype

selected_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_rs_id] if selected_rs_id else pd.DataFrame()

numeric_field_id = None
group_field_id = None

# Pick numeric field (by @id) for demonstration
if not df.empty:
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Attempt to pick a groupable (categorical) field
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() <= 20:
            group_field_id = col
            break

print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

# EDA: filter, normalize, and group
if numeric_field_id:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).all() else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, field_norm]].head())
    
    # Group by categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distribution and relationships between fields from the dataset using matplotlib.

In [ ]:
# Plot histogram of numeric field if available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Plot grouped means if group_field is available
if numeric_field_id and group_field_id and group_field_id in df.columns:
    means = df.groupby(group_field_id)[numeric_field_id].mean()
    means.plot(kind='bar', figsize=(8, 4))
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded, overviewed, and analyzed the FAIR² dataset, referencing all entities by their `@id`. Using the `mlcroissant` library, we:
- Loaded and summarized the dataset metadata.
- Listed available record sets and their fields/columns (by `@id`).
- Extracted dataframes from each record set.
- Performed filtering, normalization, and grouping on numeric fields.
- Created visualizations to explore the data distribution.

This exploration demonstrates FAIR data practices using Croissant, and provides a foundation for further clinical, biomarker, and anatomical studies on the data.